<a href="https://colab.research.google.com/github/paulinepiccio/cinema-and-war/blob/main/notebooks/03_matching_movies_conflict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import requests
import time
from google.colab import userdata
import matplotlib.pyplot as plt

API_KEY = userdata.get('TMDB_API_KEY')
BASE_URL = "https://api.themoviedb.org/3"

MOVIES_URL = "https://raw.githubusercontent.com/paulinepiccio/cinema-and-war/main/data/processed/war_movies_cleaned.csv"
CONFLICTS_URL = "https://raw.githubusercontent.com/paulinepiccio/cinema-and-war/refs/heads/main/data/conflits.csv"

df = pd.read_csv(MOVIES_URL)
conflicts = pd.read_csv(CONFLICTS_URL)

print(f"{df.shape[0]} films")
print(f"{conflicts.shape[0]} conflicts")
print(conflicts[['conflict', 'start_year', 'end_year']].to_string())

2343 films
35 conflicts
                     conflict  start_year  end_year
0                World War II        1939      1945
1           Chinese Civil War        1945      1949
2         First Indochina War        1946      1954
3             Greek Civil War        1946      1949
4                    Cold War        1947      1991
5           Malayan Emergency        1948      1960
6                  Korean War        1950      1953
7            Mau Mau Uprising        1952      1960
8                Algerian War        1954      1962
9                 Vietnam War        1955      1975
10                Suez Crisis        1956      1956
11       Hungarian Revolution        1956      1956
12                Bay of Pigs        1961      1961
13                Berlin Wall        1961      1989
14              Prague Spring        1968      1968
15  Northern Ireland Troubles        1968      1998
16             Yom Kippur War        1973      1973
17         Lebanese Civil War        197

In [3]:
#Keyword dictionary

CONFLICT_KEYWORDS = {
    'World War II': [
        'world war ii', 'world war 2', 'wwii', 'ww2', 'second world war',
        'nazi', 'nazis', 'hitler', 'third reich', 'holocaust', 'shoah',
        'd-day', 'normandy', 'pearl harbor', 'hiroshima', 'nagasaki',
        'auschwitz', 'gestapo', 'wehrmacht', 'luftwaffe', 'kamikaze',
        'french resistance', 'partisans',
        'stalingrad', 'leningrad', 'eastern front', 'pacific war',
        'iwo jima', 'okinawa', 'midway', 'el alamein', 'dunkirk',
        'battle of britain', 'the blitz', 'occupied france',
        # New keywords for better coverage
        'red army', 'soviet soldier',
        'spitfire', 'lancaster bomber', 'b-17', 'flying fortress',
        'allied forces', 'axis forces', 'wartime', 'concentration camp',
        'death camp', 'jewish refugee', 'sabotage nazi', 'collaborator',
        'occupied europe', '1944', '1943', '1942', '1941', '1940',
        # Imperial Japan-related (for Japanese cinema on WWII)
        'imperial japan', 'imperial japanese', 'japanese army',
        'manchuria', 'singapore 1942', 'philippines 1942',
    ],
    'Vietnam War': [
        'vietnam war', 'viet cong', 'vietcong', 'saigon',
        'ho chi minh', 'tet offensive', 'mekong', 'da nang', 'hanoi',
        'vietnam veteran', 'agent orange', 'napalm vietnam',
        'south vietnam', 'north vietnam',
    ],
    'Korean War': [
        'korean war', 'pyongyang', 'inchon', 'panmunjom',
        '38th parallel', 'north korean', 'south korean',
        'mash unit', 'chosin reservoir',
    ],
    'Iraq War': [
        'iraq war', 'iraqi', 'baghdad', 'fallujah', 'mosul',
        'saddam hussein', 'iraqi insurgency', 'operation iraqi freedom',
        'green zone', 'sunni insurgent',
    ],
    'War in Afghanistan': [
        'afghanistan', 'kabul', 'kandahar', 'taliban', 'al-qaeda',
        'al qaeda', 'afghan war', 'bin laden', 'helmand',
        '9/11', 'september 11', 'twin towers',
    ],
    'Soviet-Afghan War': [
        'soviet-afghan', 'soviet afghan war', 'soviet invasion of afghanistan',
        'mujahideen', 'red army afghanistan', '9th company',
    ],
    'Gulf War': [
        'gulf war', 'desert storm', 'desert shield', 'kuwait invasion',
        'operation desert', 'first gulf war',
    ],
    'Cold War': [
        'cold war', 'kgb', 'cia spy', 'east germany', 'stasi',
        'iron curtain', 'soviet spy', 'cuban missile',
        'mccarthyism', 'red scare', 'defection', 'defector',
        'east german', 'west german',
    ],
    'Algerian War': [
        'algerian war', 'algeria war', 'fln', 'battle of algiers',
        'french algeria', 'algerian independence', 'oas',
    ],
    'First Indochina War': [
        'indochina war', 'dien bien phu', 'french indochina',
    ],
    'Falklands War': [
        'falklands', 'malvinas', 'falkland islands',
    ],
    'Bosnian War': [
        'bosnia', 'bosnian war', 'sarajevo', 'srebrenica',
        'yugoslav wars', 'milosevic siege',
    ],
    'Kosovo War': [
        'kosovo war', 'kosovo conflict', 'nato bombing yugoslavia',
        'kosovo liberation',
    ],
    'Northern Ireland Troubles': [
        'northern ireland', 'belfast', 'i.r.a.', 'irish republican army',
        'the troubles', 'bloody sunday', 'ulster',
    ],
    'First Chechen War': [
        'chechnya', 'chechen', 'grozny', 'first chechen war',
    ],
    'Second Chechen War': [
        'second chechen', 'putin chechnya',
    ],
    'Syrian Civil War': [
        'syrian civil war', 'syria war', 'aleppo', 'damascus war',
        'assad regime', 'isis syria', 'syrian rebels',
    ],
    'Russo-Ukrainian War': [
        'ukraine war', 'russian invasion of ukraine', 'mariupol',
        'donbas war', 'crimea annexation', 'zelensky',
        'kyiv invasion',
    ],
    'Lebanese Civil War': [
        'lebanese civil war', 'lebanon war', 'beirut 1982',
        'first lebanon war',
    ],
    'Yom Kippur War': [
        'yom kippur', 'yom kippur war', '1973 arab-israeli',
    ],
    'Iran-Iraq War': [
        'iran-iraq', 'iran iraq war', 'iranian revolution war',
    ],
    'Rwandan Genocide': [
        'rwanda', 'rwandan genocide', 'tutsi', 'hutu', 'kigali',
    ],
    'Libyan Civil War': [
        'libya war', 'gaddafi', 'libyan civil', 'benghazi',
    ],
    'Chinese Civil War': [
        'chinese civil war', 'kuomintang', 'mao zedong war',
    ],
    'Bay of Pigs': [
        'bay of pigs', 'cuban invasion 1961', 'castro invasion',
    ],
    'Suez Crisis': [
        'suez crisis', 'suez canal war',
    ],
    'Hungarian Revolution': [
        'hungarian revolution', 'budapest 1956', '1956 hungary',
    ],
    'Prague Spring': [
        'prague spring', 'czechoslovakia 1968', 'soviet invasion czechoslovakia',
    ],
    'Russo-Georgian War': [
        'georgia war 2008', 'south ossetia war', 'russo-georgian',
    ],
    'Berlin Wall': [
        'berlin wall', 'east german escape', 'gdr escape',
    ],
    'Greek Civil War': [
        'greek civil war',
    ],
    'Malayan Emergency': [
        'malayan emergency', 'malaya communist', 'malaya 1948',
    ],
    'Mau Mau Uprising': [
        'mau mau', 'kenya uprising', 'kikuyu rebellion',
    ],
    'Yemeni Civil War': [
        'yemen war', 'yemeni civil', 'houthi',
    ],
    'War in Donbas': [
        'donbas conflict', 'donetsk war', 'luhansk',
    ],
}

In [4]:
import re

def match_conflict_from_text(text, keyword_dict):
    """
    Match a text against the keyword dictionary using WORD BOUNDARIES.
    This prevents 'ira' from matching inside 'Iraq' or 'iranian'.

    Returns the conflict with the most keyword matches.
    """
    if not isinstance(text, str) or len(text.strip()) == 0:
        return None, 0, []

    text_lower = text.lower()

    matches = {}
    for conflict, keywords in keyword_dict.items():
        matched_kws = []
        for kw in keywords:
            pattern = r'\b' + re.escape(kw) + r'\b'
            if re.search(pattern, text_lower):
                matched_kws.append(kw)
        if matched_kws:
            matches[conflict] = matched_kws

    if not matches:
        return None, 0, []

    best_conflict = max(matches, key=lambda c: len(matches[c]))
    best_matches = matches[best_conflict]

    return best_conflict, len(best_matches), best_matches


# Test
test_problematic = [
    "An Iraqi rebel fights against insurgents.",
    "An IRA bombing rocks Belfast in 1972.",
    "Iranian soldiers cross the border.",
]

print(" Testing :")
for text in test_problematic:
    conflict, n, kws = match_conflict_from_text(text, CONFLICT_KEYWORDS)
    print(f" \"{text}\"")
    print(f"   → {conflict} ({kws})\n")

 Testing :
 "An Iraqi rebel fights against insurgents."
   → Iraq War (['iraqi'])

 "An IRA bombing rocks Belfast in 1972."
   → Northern Ireland Troubles (['belfast'])

 "Iranian soldiers cross the border."
   → None ([])



In [7]:
# Apply the matching function to every movie's overview

results = df['overview'].apply(lambda x: match_conflict_from_text(x, CONFLICT_KEYWORDS))

# Unpack the tuple into 3 columns
df['matched_conflict'] = [r[0] for r in results]
df['match_score'] = [r[1] for r in results]
df['matched_keywords'] = [r[2] for r in results]

# Quick stats
n_matched = df['matched_conflict'].notna().sum()
n_unmatched = df['matched_conflict'].isna().sum()
total = len(df)

print(f"Matched : {n_matched} films ({n_matched/total*100:.1f}%)")
print(f"Unmatched : {n_unmatched} films ({n_unmatched/total*100:.1f}%)")

Matched : 1274 films (54.4%)
Unmatched : 1069 films (45.6%)


In [9]:
# Distribution of matched conflicts
conflict_counts = df['matched_conflict'].value_counts()
print(conflict_counts.to_string())

print(f"Total unique conflicts matched: {df['matched_conflict'].nunique()}")

matched_conflict
World War II                 1020
Vietnam War                    69
War in Afghanistan             65
Korean War                     29
Iraq War                       22
Bosnian War                    13
Algerian War                    8
Cold War                        8
First Chechen War               7
Gulf War                        5
Falklands War                   4
Northern Ireland Troubles       4
Rwandan Genocide                3
Russo-Ukrainian War             3
Lebanese Civil War              2
Berlin Wall                     2
War in Donbas                   2
First Indochina War             2
Mau Mau Uprising                1
Hungarian Revolution            1
Kosovo War                      1
Malayan Emergency               1
Syrian Civil War                1
Yom Kippur War                  1
Total unique conflicts matched: 24


In [11]:
# Let's verify the matching makes sense by inspecting examples
print(" Examples of matched films (5 per top conflict):")

top_conflicts = df['matched_conflict'].value_counts().head(5).index

for conflict in top_conflicts:
    print(f"{conflict}")
    examples = df[df['matched_conflict'] == conflict].sample(
        min(5, (df['matched_conflict'] == conflict).sum()),
        random_state=42
    )
    for _, row in examples.iterrows():
        print(f" {row['title']} ({row['release_year']}) — {row['matched_keywords']}")

 Examples of matched films (5 per top conflict):
World War II
 Desperado Outpost (1959) — ['second world war']
 The Thin Red Line (1964) — ['wwii']
 Father of a Soldier (1965) — ['world war ii']
 Decision Before Dawn (1951) — ['wwii', 'gestapo']
 Chronicle of Flaming Years (1961) — ['world war ii', 'wartime']
Vietnam War
 1969 (1988) — ['vietnam war']
 Apocalypse Now (1979) — ['vietnam war']
 In Love and War (1987) — ['hanoi', 'north vietnam']
 Uncommon Valor (1983) — ['vietnam war']
 Vietnam (1987) — ['vietnam war']
War in Afghanistan
 400 Bullets (2021) — ['afghanistan', 'taliban']
 Warhorse One (2023) — ['afghanistan', 'taliban']
 Rambo III (1988) — ['afghanistan']
 Leaving Afghanistan (2019) — ['afghanistan', 'afghan war']
 Septem8er Tapes (2004) — ['afghanistan', 'bin laden', '9/11']
Korean War
 Battle of Jangsari (2019) — ['korean war']
 All the Young Men (1960) — ['korean war']
 Men of the Fighting Lady (1954) — ['korean war']
 Dragonfly Squadron (1954) — ['korean war', 'south k

In [13]:
# What kind of films are unmatched?
unmatched = df[df['matched_conflict'].isna()]
print(f"{len(unmatched)} unmatched films")

print("Random sample of 15 unmatched films")
sample = unmatched.sample(min(15, len(unmatched)), random_state=42)
for _, row in sample.iterrows():
    overview_short = (row['overview'][:120] + '...') if isinstance(row['overview'], str) and len(row['overview']) > 120 else row['overview']
    print(f" {row['title']} ({row['release_year']}, {row['primary_country']})")
    print(f"   {overview_short}")

1069 unmatched films
Random sample of 15 unmatched films
 Houston: The Legend of Texas (1986, US)
   Sam Elliot stars as Sam Houston, the visionary who nearly single-handedly forged the state of Texas into a powerful enti...
 All the Queen's Men (2001, US)
   A mismatched team of British Special Services agents led by an American must infiltrate, in disguise, a female-run Enigm...
 The Guardians (2017, FR)
   In this ensemble drama set in rural France, the women of the Paridier farm are left to run it by themselves while their ...
 Henry V (1989, GB)
   In 1415, in the midst of the Hundred Years' War, the young King Henry V of England embarks on the conquest of France.
 Uncanny Valley (2015, FR)
   Inside a museum, nowadays. A diorama represents two young soldiers in the trenches. All of a sudden, we are thrown into ...
 The Horse Soldiers (1959, US)
   A Union Cavalry outfit is sent behind confederate lines in strength to destroy a rail supply center. Along with them is ...
 Courage U

In [14]:
# Look for potentially weird matches (low match score might indicate false positive)
print("Low-confidence matches (only 1 keyword found) :")
low_confidence = df[(df['match_score'] == 1) & (df['matched_conflict'].notna())]
print(f"Total low-confidence: {len(low_confidence)}\n")

sample = low_confidence.sample(min(10, len(low_confidence)), random_state=42)
for _, row in sample.iterrows():
    print(f" {row['title']} ({row['release_year']}) → {row['matched_conflict']}")
    print(f"   Keyword found: {row['matched_keywords']}")
    print(f"   Overview: {row['overview'][:150]}...")

Low-confidence matches (only 1 keyword found) :
Total low-confidence: 831

 Forbidden Films (2014) → World War II
   Keyword found: ['allied forces']
   Overview: Between 1933 and 1945 roughly 1200 films were made in Germany, of which 300 were banned by the Allied forces. Today, around 40 films, called "Vorbehal...
 My Dead Friend Zoe (2024) → War in Afghanistan
   Keyword found: ['afghanistan']
   Overview: Merit, a U.S. Army veteran suffering from PTSD, is repeatedly tortured by visions of her deceased friend and company buddy Zoe. After her Afghanistan ...
 The Password Is Courage (1962) → World War II
   Keyword found: ['world war ii']
   Overview: Sergeant-Major Charles Coward, a brave British soldier is captured by German forces during World War II. When he's thrown into a prisoner of war camp,...
 Seeds of Destiny (1946) → World War II
   Keyword found: ['nazi']
   Overview: Oscar winning postwar propaganda film in support of the United Nations Relief and Rehabilitation Administ

In [18]:
# Mapping between TMDB official keyword names and our conflicts

TMDB_KEYWORD_MAPPING = {
    'World War II': [
        'world war ii', 'wwii', 'second world war',
        'nazism', 'nazi germany', 'holocaust', 'shoah',
        'normandy invasion', 'd-day', 'pearl harbor',
        'hiroshima', 'nagasaki', 'atomic bomb',
        'french resistance', 'resistance', 'partisan',
        'pacific war', 'eastern front', 'battle of britain',
        'concentration camp', 'jewish persecution',
        'imperial japan', 'kamikaze',
    ],
    'Vietnam War': [
        'vietnam war', 'vietnam', 'viet cong', 'vietnam veteran',
    ],
    'Korean War': [
        'korean war', 'korea',
    ],
    'Iraq War': [
        'iraq war', 'iraq', 'baghdad',
    ],
    'War in Afghanistan': [
        'afghanistan war', 'war in afghanistan', 'taliban',
        'al-qaeda', 'osama bin laden',
    ],
    'Soviet-Afghan War': [
        'soviet-afghan war', 'soviet war in afghanistan', 'mujahideen',
    ],
    'Gulf War': [
        'gulf war', 'persian gulf war', 'operation desert storm',
    ],
    'Cold War': [
        'cold war', 'kgb', 'soviet union', 'east germany',
        'stasi', 'iron curtain', 'cuban missile crisis',
    ],
    'Algerian War': [
        'algerian war', 'algeria',
    ],
    'First Indochina War': [
        'first indochina war', 'indochina war',
    ],
    'Falklands War': [
        'falklands war', 'falklands',
    ],
    'Bosnian War': [
        'bosnian war', 'yugoslav wars', 'sarajevo',
    ],
    'Kosovo War': [
        'kosovo war', 'kosovo',
    ],
    'Northern Ireland Troubles': [
        'northern ireland', 'the troubles', 'ira',
        'belfast', 'ulster',
    ],
    'First Chechen War': [
        'chechen war', 'first chechen war', 'chechnya',
    ],
    'Second Chechen War': [
        'second chechen war',
    ],
    'Syrian Civil War': [
        'syrian civil war', 'syria',
    ],
    'Russo-Ukrainian War': [
        'russo-ukrainian war', 'russian invasion of ukraine',
        'ukraine war',
    ],
    'Lebanese Civil War': [
        'lebanese civil war', 'lebanon',
    ],
    'Yom Kippur War': [
        'yom kippur war', 'arab-israeli conflict',
    ],
    'Iran-Iraq War': [
        'iran-iraq war',
    ],
    'Rwandan Genocide': [
        'rwandan genocide', 'rwanda',
    ],
    'Libyan Civil War': [
        'libyan civil war', 'libya',
    ],
    'Chinese Civil War': [
        'chinese civil war',
    ],
    'Bay of Pigs': [
        'bay of pigs invasion',
    ],
    'Suez Crisis': [
        'suez crisis',
    ],
    'Hungarian Revolution': [
        'hungarian revolution',
    ],
    'Prague Spring': [
        'prague spring',
    ],
    'Russo-Georgian War': [
        'russo-georgian war',
    ],
    'Berlin Wall': [
        'berlin wall',
    ],
    'Greek Civil War': [
        'greek civil war',
    ],
    'Malayan Emergency': [
        'malayan emergency',
    ],
    'Mau Mau Uprising': [
        'mau mau uprising',
    ],
    'Yemeni Civil War': [
        'yemeni civil war', 'yemen',
    ],
    'War in Donbas': [
        'war in donbas',
    ],
}

print(f"TMDB keyword mapping: {len(TMDB_KEYWORD_MAPPING)} conflicts")
print(f"Total TMDB keywords: {sum(len(kws) for kws in TMDB_KEYWORD_MAPPING.values())}")

TMDB keyword mapping: 35 conflicts
Total TMDB keywords: 97


In [15]:
def fetch_movie_keywords(movie_id):
    """
    Fetch keywords for a specific movie from TMDB API.

    Args:
        movie_id (int): TMDB movie ID

    Returns:
        list: list of keyword names (lowercased), or [] if error/none
    """
    url = f"{BASE_URL}/movie/{movie_id}/keywords"
    params = {'api_key': API_KEY}

    try:
        response = requests.get(url, params=params, timeout=10)
        if response.status_code != 200:
            return []

        data = response.json()
        keywords = data.get('keywords', [])
        return [kw['name'].lower() for kw in keywords]
    except Exception as e:
        print(f"Error for id={movie_id}: {e}")
        return []


# Quick test on a well-known movie: Saving Private Ryan (TMDB id: 857)
print("Test on Saving Private Ryan (id=857):")
test_keywords = fetch_movie_keywords(857)
print(f"Keywords found: {test_keywords}")
print(f"Total: {len(test_keywords)}")

Test on Saving Private Ryan (id=857):
Keywords found: ['dying and death', 'self sacrifice', 'bravery', 'world war ii', 'duty', 'normandy, france', 'troops', 'waffen ss', 'omaha beach', 'rescue mission', 'cowardice', 'us army', 'based on true story', 'd-day', 'military', 'german soldier', 'military operation', '1940s', 'bloody deaths', 'u.s. army soldier', 'u.s. army ranger', 'massive casualties']
Total: 22


In [19]:
def match_conflict_from_tmdb_keywords(tmdb_keywords, keyword_mapping):
    """
    Match a list of TMDB keywords against the conflict mapping.
    Returns the conflict with the most matches.
    """
    if not tmdb_keywords:
        return None, 0, []

    matches = {}
    for conflict, target_keywords in keyword_mapping.items():
        matched_kws = [kw for kw in target_keywords if kw in tmdb_keywords]
        if matched_kws:
            matches[conflict] = matched_kws

    if not matches:
        return None, 0, []

    best_conflict = max(matches, key=lambda c: len(matches[c]))
    best_matches = matches[best_conflict]

    return best_conflict, len(best_matches), best_matches


# Test on Saving Private Ryan
conflict, n, matched = match_conflict_from_tmdb_keywords(test_keywords, TMDB_KEYWORD_MAPPING)
print(f"Test: Saving Private Ryan  {conflict}")
print(f"Matched keywords : {matched}")
print(f"Score: {n}")

Test: Saving Private Ryan  World War II
Matched keywords : ['world war ii', 'd-day']
Score: 2


In [20]:
# Get IDs of films not yet matched by Layer 1
unmatched_ids = df[df['matched_conflict'].isna()]['id'].tolist()
total_to_fetch = len(unmatched_ids)
print(f"Films to query via API: {total_to_fetch}")

movie_keywords_dict = {}
errors = []

# Fetch keywords with progress indicator
for i, movie_id in enumerate(unmatched_ids):
    keywords = fetch_movie_keywords(movie_id)
    movie_keywords_dict[movie_id] = keywords

    if not keywords:
        errors.append(movie_id)

    # Progress every 50 films
    if (i + 1) % 50 == 0:
        elapsed_pct = (i + 1) / total_to_fetch * 100

    # Respect TMDB rate limit (50 req/sec, we go much slower for safety)
    time.sleep(0.1)

# Final stats
films_with_keywords = sum(1 for kws in movie_keywords_dict.values() if kws)
films_no_keywords = len(movie_keywords_dict) - films_with_keywords

print(f"Films processed: {len(movie_keywords_dict)}")
print(f"Films with at least 1 keyword: {films_with_keywords}")
print(f"Films with NO keywords on TMDB: {films_no_keywords}")

Films to query via API: 1069
Films processed: 1069
Films with at least 1 keyword: 793
Films with NO keywords on TMDB: 276


In [21]:
# Initialize match_source column to track which layer matched each film
if 'match_source' not in df.columns:
    df['match_source'] = None

# Mark existing layer 1 matches
df.loc[df['matched_conflict'].notna() & df['match_source'].isna(), 'match_source'] = 'overview'

# Apply TMDB keyword matching
print("Matching TMDB keywords against our conflict mapping...")

n_newly_matched = 0
new_matches_by_conflict = {}

for movie_id, keywords in movie_keywords_dict.items():
    conflict, n, matched = match_conflict_from_tmdb_keywords(keywords, TMDB_KEYWORD_MAPPING)

    if conflict:
        # Update the dataframe
        df.loc[df['id'] == movie_id, 'matched_conflict'] = conflict
        df.loc[df['id'] == movie_id, 'match_score'] = n
        df.loc[df['id'] == movie_id, 'matched_keywords'] = str(matched)
        df.loc[df['id'] == movie_id, 'match_source'] = 'tmdb_keywords'
        n_newly_matched += 1

        new_matches_by_conflict[conflict] = new_matches_by_conflict.get(conflict, 0) + 1

# Final stats
print(f"Newly matched by TMDB keywords: {n_newly_matched} films")

print("New matches by conflict (Layer 2 only):")
for conflict, count in sorted(new_matches_by_conflict.items(), key=lambda x: -x[1]):
    print(f"   {conflict}: +{count}")

# Overall stats
total_matched = df['matched_conflict'].notna().sum()
total = len(df)

print(f"Final matching results (Layer 1 + Layer 2):")
print(f"Total matched: {total_matched}/{total} ({total_matched/total*100:.1f}%)")
print(f"Still unmatched: {total - total_matched}/{total} ({(total - total_matched)/total*100:.1f}%)")

print(f"Match source breakdown:")
print(df['match_source'].value_counts(dropna=False))

Matching TMDB keywords against our conflict mapping...
Newly matched by TMDB keywords: 172 films
New matches by conflict (Layer 2 only):
   World War II: +90
   Vietnam War: +38
   Cold War: +16
   Iraq War: +11
   Syrian Civil War: +4
   Gulf War: +3
   Russo-Ukrainian War: +3
   First Chechen War: +2
   Algerian War: +2
   Falklands War: +1
   Libyan Civil War: +1
   Korean War: +1
Final matching results (Layer 1 + Layer 2):
Total matched: 1446/2343 (61.7%)
Still unmatched: 897/2343 (38.3%)
Match source breakdown:
match_source
overview         1274
None              897
tmdb_keywords     172
Name: count, dtype: int64


In [22]:
# Full distribution of conflicts after Layer 1 + Layer 2
print("Top conflicts after Layer 1 + Layer 2:")
conflict_counts = df['matched_conflict'].value_counts()
print(conflict_counts.to_string())

print(f"Total unique conflicts matched: {df['matched_conflict'].nunique()}")

# Also show distribution by source
print(f"Conflicts by source (overview vs tmdb_keywords):")
source_breakdown = df.groupby(['matched_conflict', 'match_source']).size().unstack(fill_value=0)
print(source_breakdown.to_string())

Top conflicts after Layer 1 + Layer 2:
matched_conflict
World War II                 1110
Vietnam War                   107
War in Afghanistan             65
Iraq War                       33
Korean War                     30
Cold War                       24
Bosnian War                    13
Algerian War                   10
First Chechen War               9
Gulf War                        8
Russo-Ukrainian War             6
Syrian Civil War                5
Falklands War                   5
Northern Ireland Troubles       4
Rwandan Genocide                3
Berlin Wall                     2
Lebanese Civil War              2
First Indochina War             2
War in Donbas                   2
Hungarian Revolution            1
Mau Mau Uprising                1
Kosovo War                      1
Libyan Civil War                1
Malayan Emergency               1
Yom Kippur War                  1
Total unique conflicts matched: 25
Conflicts by source (overview vs tmdb_keywords):
match_sour

In [24]:
# Categorize the 827 unmatched films to understand WHAT they are
# Goal: separate out-of-scope films (which shouldn't count) from genuinely missed classifications

OUT_OF_SCOPE_KEYWORDS = {
    'World War I': [
        'world war i', 'wwi', 'ww1', 'first world war',
        'the great war', 'trench warfare', 'somme', 'verdun',
        'gallipoli', 'kaiser wilhelm', 'doughboy',
        '1914', '1915', '1916', '1917', '1918',
    ],
    'American Civil War': [
        'american civil war', 'confederate army', 'confederate soldier',
        'union army', 'union soldier', 'gettysburg', 'antietam',
        'abraham lincoln', 'civil war 1861', 'slavery war',
        '1861', '1862', '1863', '1864', '1865',
    ],
    'Pre-modern conflicts': [
        'viking', 'medieval', 'crusade', 'roman empire', 'roman legion',
        'spartan warrior', 'samurai', 'shogun', 'knight templar',
        'napoleonic war', 'napoleon bonaparte', 'waterloo',
        'joan of arc', 'hundred years war', 'caesar',
        'genghis khan', 'mongol invasion',
    ],
    'Fictional / Sci-fi': [
        'alien invasion', 'zombie apocalypse', 'robot war',
        'galactic war', 'mecha', 'gundam', 'futuristic',
        'space marine', 'cyborg', 'post-apocalyptic',
        'time travel war', 'mutant', 'star ship',
    ],
    'Training / Military life (no specific conflict)': [
        'boot camp', 'military academy', 'basic training',
        'recruit', 'drill sergeant', 'enlist',
    ],
}


def detect_out_of_scope(text):
    """Returns the out-of-scope category if matched, else None."""
    if not isinstance(text, str):
        return None
    text_lower = text.lower()
    for category, keywords in OUT_OF_SCOPE_KEYWORDS.items():
        for kw in keywords:
            pattern = r'\b' + re.escape(kw) + r'\b'
            if re.search(pattern, text_lower):
                return category
    return None


# Apply to unmatched films only
mask_unmatched = df['matched_conflict'].isna()
df.loc[mask_unmatched, 'out_of_scope_category'] = df.loc[mask_unmatched, 'overview'].apply(detect_out_of_scope)

# Stats
n_out_of_scope = df['out_of_scope_category'].notna().sum()
n_truly_unmatched = mask_unmatched.sum() - n_out_of_scope

print(f"Categorization of 827 unmatched films:")
print(f"Identified as out-of-scope: {n_out_of_scope}")
print(df['out_of_scope_category'].value_counts())

print(f"Genuinely unmatched (within scope but unidentified): {n_truly_unmatched}")

Categorization of 827 unmatched films:
Identified as out-of-scope: 157
out_of_scope_category
World War I                                        86
Pre-modern conflicts                               28
American Civil War                                 26
Fictional / Sci-fi                                 13
Training / Military life (no specific conflict)     4
Name: count, dtype: int64
Genuinely unmatched (within scope but unidentified): 740


In [25]:
# Inspect the genuinely unmatched films

genuinely_unmatched = df[
    (df['matched_conflict'].isna()) &
    (df['out_of_scope_category'].isna())
].copy()

print(f"{len(genuinely_unmatched)} genuinely unmatched films")

# Distribution by country
print("Distribution by country:")
print(genuinely_unmatched['primary_country'].value_counts())

# Distribution by decade
print("Distribution by decade:")
print(genuinely_unmatched['decade'].value_counts().sort_index())

# Sample of films to inspect
print("Random sample of 20 unmatched films:")
sample = genuinely_unmatched.sample(min(20, len(genuinely_unmatched)), random_state=42)
for _, row in sample.iterrows():
    overview_short = row['overview'][:150] + '...' if len(row['overview']) > 150 else row['overview']
    print(f" {row['title']} ({row['release_year']}, {row['primary_country']})")
    print(f"   {overview_short}")

740 genuinely unmatched films
Distribution by country:
primary_country
US    304
GB    136
RU    102
FR    100
JP     66
DE     32
Name: count, dtype: int64
Distribution by decade:
decade
1940     31
1950    114
1960    100
1970     54
1980     69
1990     67
2000    112
2010    129
2020     64
Name: count, dtype: int64
Random sample of 20 unmatched films:
 Hornblower: Mutiny (2001, GB)
   Hornblower and his comrades come under the command of a revered but mentally unstable captain and are forced to mutiny in order to save their ship, th...
 A Legend or Was It? (1963, JP)
   A Tokyo family escaping the war relocates to a Hokkaido village; their daughter is set to marry the local leader's son, but her siblings disapprove.
 The Tree of Guernica (1975, FR)
   The fictional town of Villa Romero is the set upon which the events of Spain's civil war play out. Villa Romero is home to Vandale (Mariangela Melato)...
 Uncivil War Birds (1946, US)
   The stooges are civil war soldiers who are con

In [26]:
total_films = len(df)
in_scope_films = total_films - n_out_of_scope
matched_films = df['matched_conflict'].notna().sum()

print(f"Final statistics:")
print(f"Total films in dataset: {total_films}")
print(f"Out-of-scope films: {n_out_of_scope} ({n_out_of_scope/total_films*100:.1f}%)")
print(f"In-scope films: {in_scope_films}")
print(f"Matched (in-scope only): {matched_films}/{in_scope_films} ({matched_films/in_scope_films*100:.1f}%)")
print(f"Genuinely unmatched: {in_scope_films - matched_films}")

Final statistics:
Total films in dataset: 2343
Out-of-scope films: 157 (6.7%)
In-scope films: 2186
Matched (in-scope only): 1446/2186 (66.1%)
Genuinely unmatched: 740


In [27]:
# Save the final classified dataset
output_path = 'war_movies_classified.csv'
df.to_csv(output_path, index=False, encoding='utf-8')
print(f" Saved: {output_path}")
print(f" Final size: {df.shape[0]} films, {df.shape[1]} columns")

# Show columns
print(f"Final columns:")
for col in df.columns:
    n_filled = df[col].notna().sum()
    print(f"   {col}: {n_filled} non-null values")

 Saved: war_movies_classified.csv
 Final size: 2343 films, 20 columns
Final columns:
   id: 2343 non-null values
   title: 2343 non-null values
   original_title: 2343 non-null values
   original_language: 2343 non-null values
   release_date: 2343 non-null values
   release_year: 2343 non-null values
   decade: 2343 non-null values
   overview: 2343 non-null values
   popularity: 2343 non-null values
   vote_average: 2343 non-null values
   vote_count: 2343 non-null values
   origin_country_code: 2343 non-null values
   origin_country_name: 2343 non-null values
   genre_ids: 2343 non-null values
   primary_country: 2343 non-null values
   matched_conflict: 1446 non-null values
   match_score: 2343 non-null values
   matched_keywords: 2343 non-null values
   match_source: 1446 non-null values
   out_of_scope_category: 157 non-null values


In [28]:
# Detect films released BEFORE their matched conflict started (impossible)

conflict_dates = conflicts.set_index('conflict')[['start_year', 'end_year']].to_dict('index')


def check_temporal_consistency(row):
    """
    A film should be released DURING or AFTER its conflict, never significantly before.
    We allow a 2-year buffer for edge cases (films "in anticipation" are extremely rare).
    """
    conflict = row['matched_conflict']
    if pd.isna(conflict) or conflict not in conflict_dates:
        return 'no_check_needed'

    release_year = row['release_year']
    conflict_start = conflict_dates[conflict]['start_year']

    if release_year < conflict_start - 2:
        return 'temporal_inconsistency'
    return 'consistent'


df['temporal_check'] = df.apply(check_temporal_consistency, axis=1)

# Display inconsistencies
inconsistencies = df[df['temporal_check'] == 'temporal_inconsistency']
print(f"Found {len(inconsistencies)} temporally inconsistent matches")

if len(inconsistencies) > 0:
    print("Examples (films released BEFORE their matched conflict):")
    cols_to_show = ['title', 'release_year', 'matched_conflict', 'matched_keywords', 'primary_country']
    print(inconsistencies[cols_to_show].head(20).to_string())

Found 11 temporally inconsistent matches
Examples (films released BEFORE their matched conflict):
                             title  release_year    matched_conflict      matched_keywords primary_country
38                       Rambo III          1988  War in Afghanistan         [afghanistan]              US
242                      The Beast          1988  War in Afghanistan         [afghanistan]              US
965        The Brigand of Kandahar          1965  War in Afghanistan            [kandahar]              GB
1023              Afghan Breakdown          1991  War in Afghanistan         [afghanistan]              RU
1139              The Human Shield          1992            Iraq War               [iraqi]              US
1169                Bravo Two Zero          1999            Iraq War      [saddam hussein]              GB
1636                Peshawar Waltz          1994  War in Afghanistan         [afghanistan]              RU
1667           Hot Summer in Kabul          19

In [30]:
# inspect 30 films stratified by conflict to estimate accuracy

import random

sample_size = 100
matched_only = df[df['matched_conflict'].notna()].copy()

# Stratified sampling across top conflicts
top_conflicts = matched_only['matched_conflict'].value_counts().head(10).index.tolist()
sample_dfs = []
for conflict in top_conflicts:
    subset = matched_only[matched_only['matched_conflict'] == conflict]
    sample_dfs.append(subset.sample(min(3, len(subset)), random_state=42))

audit_sample = pd.concat(sample_dfs).reset_index(drop=True)

print(f"AUDIT SAMPLE — {len(audit_sample)} films to verify manually:")

for idx, row in audit_sample.iterrows():
    print(f"[{idx+1}]  {row['title']} ({row['release_year']}, {row['primary_country']})")
    print(f"→ Matched to: {row['matched_conflict']}")
    print(f"→ Keywords found: {row['matched_keywords']}")
    print(f"→ Match source: {row['match_source']}")
    overview_short = row['overview'][:250] + '...' if len(str(row['overview'])) > 250 else row['overview']
    print(f"→ Overview: {overview_short}")

AUDIT SAMPLE — 30 films to verify manually:
[1]  And We Had Silence... (1977, RU)
→ Matched to: World War II
→ Keywords found: ['wwii']
→ Match source: overview
→ Overview: An adult man is recalling his childhood years in North Russia during WWII.
[2]  McHale's Navy Joins the Air Force (1965, US)
→ Matched to: World War II
→ Keywords found: ['world war ii']
→ Match source: tmdb_keywords
→ Overview: The crew of PT-73 are in trouble again when Ensign Parker is mistaken for a pilot and gets shanghied into the Air Force.
[3]  Operation Petticoat (1959, US)
→ Matched to: World War II
→ Keywords found: ['world war ii']
→ Match source: overview
→ Overview: A World War II submarine commander finds himself stuck with a damaged sub, a con-man executive officer, and a group of army nurses.
[4]  The Bamboo Incident (1970, FR)
→ Matched to: Vietnam War
→ Keywords found: ['viet cong']
→ Match source: overview
→ Overview: A young Vietnamese boy's life is thrown into turmoil by the war raging in his c

In [32]:
# Reclassify the 5 misclassified Soviet-Afghan films
# Films released 1979-1989 with 'afghanistan' keyword likely refer to Soviet-Afghan War

reclassify_to_soviet_afghan = [
    'Rambo III',
    'The Beast',
    'Peshawar Waltz',
    'Afgan: The Soviet Experience',
]

for title in reclassify_to_soviet_afghan:
    mask = df['title'] == title
    df.loc[mask, 'matched_conflict'] = 'Soviet-Afghan War'
    df.loc[mask, 'match_source'] = 'manual_correction'
    print(f"Reclassified: {title} → Soviet-Afghan War")

# The Brigand of Kandahar is genuinely out of scope (1880, British-Afghan war)
df.loc[df['title'] == 'The Brigand of Kandahar', 'matched_conflict'] = None
df.loc[df['title'] == 'The Brigand of Kandahar', 'match_source'] = 'rejected_out_of_scope'
df.loc[df['title'] == 'The Brigand of Kandahar', 'out_of_scope_category'] = 'Pre-modern conflicts'
# The Human Shield (1992) and Bravo Two Zero (1999) refer to Gulf War, not Iraq War
df.loc[df['title'] == 'The Human Shield', 'matched_conflict'] = 'Gulf War'
df.loc[df['title'] == 'The Human Shield', 'match_source'] = 'manual_correction'
df.loc[df['title'] == 'Bravo Two Zero', 'matched_conflict'] = 'Gulf War'
df.loc[df['title'] == 'Bravo Two Zero', 'match_source'] = 'manual_correction'
# Re-run the temporal check to confirm
df['temporal_check'] = df.apply(check_temporal_consistency, axis=1)
remaining = df[df['temporal_check'] == 'temporal_inconsistency']

Reclassified: Rambo III → Soviet-Afghan War
Reclassified: The Beast → Soviet-Afghan War
Reclassified: Peshawar Waltz → Soviet-Afghan War
Reclassified: Afgan: The Soviet Experience → Soviet-Afghan War


In [33]:
# Save final dataset
output_path = 'war_movies_classified.csv'
df.to_csv(output_path, index=False, encoding='utf-8')
print(f" Saved: {output_path}")

# Compute final summary statistics dynamically
total_films = len(df)
n_matched = df['matched_conflict'].notna().sum()
n_out_of_scope = df['out_of_scope_category'].notna().sum()
n_unmatched = total_films - n_matched - n_out_of_scope

# Note: a film could in theory be both matched AND out_of_scope flagged
# Let's verify the categories are mutually exclusive
overlap = ((df['matched_conflict'].notna()) & (df['out_of_scope_category'].notna())).sum()

print(f"FINAL DATASET SUMMARY")
print(f"Total films: {total_films}")
print(f"Matched to a conflict: {n_matched} ({n_matched/total_films*100:.1f}%)")
print(f"Out-of-scope: {n_out_of_scope} ({n_out_of_scope/total_films*100:.1f}%)")
print(f"Unmatched (genuinely): {n_unmatched} ({n_unmatched/total_films*100:.1f}%)")

# Match source breakdown
print(f"Match source breakdown:")
print(df['match_source'].value_counts(dropna=False).to_string())

# Top conflicts
print(f"Top 10 conflicts matched:")
print(df['matched_conflict'].value_counts().head(10).to_string())

# Effective matching rate (excluding out-of-scope)
in_scope = total_films - n_out_of_scope
effective_rate = n_matched / in_scope * 100
print(f"Effective matching rate (excluding out-of-scope):")
print(f"{n_matched} / {in_scope} = {effective_rate:.1f}%")

 Saved: war_movies_classified.csv
FINAL DATASET SUMMARY
Total films: 2343
Matched to a conflict: 1445 (61.7%)
Out-of-scope: 158 (6.7%)
Unmatched (genuinely): 740 (31.6%)
Match source breakdown:
match_source
overview                 1267
None                      897
tmdb_keywords             172
manual_correction           6
rejected_out_of_scope       1
Top 10 conflicts matched:
matched_conflict
World War II          1110
Vietnam War            107
War in Afghanistan      60
Iraq War                31
Korean War              30
Cold War                24
Bosnian War             13
Gulf War                10
Algerian War            10
First Chechen War        9
Effective matching rate (excluding out-of-scope):
1445 / 2185 = 66.1%


In [34]:
from google.colab import files
files.download('war_movies_classified.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>